In [1]:
import os
import json as _json
from pathlib import Path
from dataclasses import dataclass, field
from typing import cast
from typing import List, Optional, Dict, Any, Tuple, Union, Callable, Set
import pandas as pd
import polars as pl
import io
import numpy as np
from types import SimpleNamespace
from polars.testing import assert_frame_equal as pl_assert_frame_equal
print('pandas:', pd.__version__, ' polars:', pl.__version__)

/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/envs/my_nlp_env/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


pandas: 3.0.2  polars: 1.39.3


In [2]:
# ── Fixtures ────────────────────────────────────────────────────────────────

# --- measured_data_pivot_join ---
FIX_MEASURED_DATA_PIVOT_JOIN_INDEX_VALS = [("OBS_A", "2020-01-01"), ("OBS_B", "2020-01-02")]
FIX_MEASURED_DATA_PIVOT_JOIN_OBS_VALS = np.array([1.0, 2.0])
FIX_MEASURED_DATA_PIVOT_JOIN_REALIZATIONS = np.array([0, 1, 2])
FIX_MEASURED_DATA_PIVOT_JOIN_RESP_VALS = np.array([[1.1, 2.1], [0.9, 1.9], [1.2, 2.2]])
FIX_MEASURED_DATA_PIVOT_JOIN_STD_VALS = np.array([0.1, 0.2])

print("✅ Fixtures loaded")


✅ Fixtures loaded


In [3]:
# ── Before wrappers (verbatim pandas) ───────────────────────────────────────

def before_measured_data_pivot_join(index_vals, obs_vals, realizations, resp_vals, std_vals):
    n_real = len(realizations)
    data = np.vstack([obs_vals, std_vals, resp_vals.reshape(n_real, -1)])
    return pd.DataFrame(
    data,
    index=("OBS", "STD", *realizations),
    columns=pd.MultiIndex.from_tuples(index_vals),
    )
    return columns

In [4]:
# ── Generated wrappers (verbatim LLM-generated Polars) ──────────────────────

def gen_measured_data_pivot_join(index_vals, obs_vals, realizations, resp_vals, std_vals):
    import numpy as np

    n_real = len(realizations)
    data = np.vstack([obs_vals, std_vals, resp_vals.reshape(n_real, -1)])
    return (
        pl.DataFrame(
            data,
            schema=[str(col) for col in index_vals],
            orient="row",
        )
        .with_columns(pl.Series("index", ["OBS", "STD", *realizations]))
        .select(["index", *[str(col) for col in index_vals]])
    )

In [5]:
# ── Comparison helper ───────────────────────────────────────────────────────
def _index_is_trivial(idx):
    # Unnamed + integer-valued covers both a fresh RangeIndex and the leftover
    # positional index after filtering/boolean-masking a RangeIndex-based frame
    # (pandas downgrades RangeIndex to a plain Int64Index on filter, but it's
    # still just leftover row positions, not real data). A set_index(...)
    # always carries the original column's name, so any genuinely meaningful
    # index is caught by the "name is not None" branch.
    return idx.name is None and pd.api.types.is_integer_dtype(idx.dtype)


def _to_pl(r):
    if isinstance(r, pl.DataFrame): return r
    if isinstance(r, pd.DataFrame): return pl.from_pandas(r.reset_index(drop=True) if _index_is_trivial(r.index) else r.reset_index())
    if isinstance(r, pd.Series): return pl.from_pandas(r.to_frame().reset_index(drop=True) if _index_is_trivial(r.index) else r.to_frame().reset_index())
    return None

def compare(before_result, gen_result, label, check_row_order=False):
    raw_label = str(label)
    label_parts = raw_label.strip().split()
    is_l3 = bool(label_parts and label_parts[0].upper() == "L3")
    layer = "L3" if is_l3 else "L2"
    kind = "edge" if is_l3 else "equivalence"
    if is_l3:
        label_parts = label_parts[1:]
        if label_parts and label_parts[0].lower() in ("edge", "branch"):
            label_parts = label_parts[1:]
        display_label = " ".join(label_parts)
    else:
        display_label = raw_label

    left  = _to_pl(before_result.collect() if isinstance(before_result, pl.LazyFrame) else before_result)
    right = _to_pl(gen_result.collect() if isinstance(gen_result, pl.LazyFrame) else gen_result)
    if left is None and right is None:
        print(f"⚠️  {layer} {kind} {display_label}: both sides non-DataFrame (no output to compare)")
        return
    if left is None or right is None:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — one side returned DataFrame, other did not")
        return
    left_cols, right_cols = set(left.columns), set(right.columns)
    if left_cols != right_cols:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — column sets differ (before-only={left_cols - right_cols}, gen-only={right_cols - left_cols})")
        return
    common = list(left.columns)
    try:
        pl_assert_frame_equal(left.select(common), right.select(common),
                              check_dtypes=False, check_row_order=check_row_order)
        print(f"✅ {layer} {kind} {display_label}: MATCH")
    except Exception as e:
        print(f"❌ {layer} {kind} {display_label}: MISMATCH — {e}")


In [6]:

# === Tests: measured_data_pivot_join ===

def _expected_measured(frame):
    return pl.DataFrame({
        "row": [str(v) for v in frame.index],
        **{str(col): frame[col].to_list() for col in frame.columns},
    })

def _normalise_generated_measured(frame):
    if not isinstance(frame, pl.DataFrame): return frame
    if "" in frame.columns: frame = frame.rename({"": "row"})
    return frame.with_columns(pl.col("row").cast(pl.String)) if "row" in frame.columns else frame

def _run_measured(label, index_vals, obs_vals, realizations, resp_vals, std_vals):
    rb = before_measured_data_pivot_join(index_vals, obs_vals, realizations, resp_vals, std_vals)
    rg = gen_measured_data_pivot_join(index_vals, obs_vals, realizations, resp_vals, std_vals)
    compare(_expected_measured(rb), _normalise_generated_measured(rg), label, check_row_order=True)

try:
    _r = gen_measured_data_pivot_join(FIX_MEASURED_DATA_PIVOT_JOIN_INDEX_VALS, FIX_MEASURED_DATA_PIVOT_JOIN_OBS_VALS, FIX_MEASURED_DATA_PIVOT_JOIN_REALIZATIONS, FIX_MEASURED_DATA_PIVOT_JOIN_RESP_VALS, FIX_MEASURED_DATA_PIVOT_JOIN_STD_VALS)
    print("✅ L1 smoke gen_measured_data_pivot_join: OK, type=", type(_r).__name__)
except Exception as _e:
    print(f"❌ L1 smoke gen_measured_data_pivot_join: {type(_e).__name__}: {_e}")

try:
    before_measured_data_pivot_join(FIX_MEASURED_DATA_PIVOT_JOIN_INDEX_VALS, FIX_MEASURED_DATA_PIVOT_JOIN_OBS_VALS, FIX_MEASURED_DATA_PIVOT_JOIN_REALIZATIONS, FIX_MEASURED_DATA_PIVOT_JOIN_RESP_VALS, FIX_MEASURED_DATA_PIVOT_JOIN_STD_VALS)
    print("✅ L1 smoke before_measured_data_pivot_join: OK")
except Exception as _e:
    print(f"❌ L1 smoke before_measured_data_pivot_join: {type(_e).__name__}: {_e}")

try:
    _run_measured("measured_data_pivot_join labels and values", FIX_MEASURED_DATA_PIVOT_JOIN_INDEX_VALS, FIX_MEASURED_DATA_PIVOT_JOIN_OBS_VALS, FIX_MEASURED_DATA_PIVOT_JOIN_REALIZATIONS, FIX_MEASURED_DATA_PIVOT_JOIN_RESP_VALS, FIX_MEASURED_DATA_PIVOT_JOIN_STD_VALS)
except Exception as _e:
    print(f"❌ L2 equivalence measured_data_pivot_join: setup error — {type(_e).__name__}: {_e}")

try:
    _run_measured("L3 edge measured_data_pivot_join single realization", [("OBS_A", "2020-01-01")], np.array([5.0]), np.array([7]), np.array([[5.5]]), np.array([0.5]))
except Exception as _e:
    print(f"❌ L3 edge measured_data_pivot_join: {type(_e).__name__}: {_e}")


❌ L1 smoke gen_measured_data_pivot_join: TypeError: unexpected value while building Series of type String; found value of type Int64: 0

Hint: Try setting `strict=False` to allow passing data with mixed types.
✅ L1 smoke before_measured_data_pivot_join: OK
❌ L2 equivalence measured_data_pivot_join: setup error — TypeError: unexpected value while building Series of type String; found value of type Int64: 0

Hint: Try setting `strict=False` to allow passing data with mixed types.
❌ L3 edge measured_data_pivot_join: TypeError: unexpected value while building Series of type String; found value of type Int64: 7

Hint: Try setting `strict=False` to allow passing data with mixed types.
